In [1]:
import os
import pandas as pd
import numpy as np
import pickle, gzip
from ecfp import generate_ecfp6_df
from physico import generate_physico_df
from toxico import generate_toxico_df
from cellline import process_cellline_matrix
project_folder = os.path.dirname(os.path.abspath("__file__"))
data_folder = os.path.join(project_folder, "dataset")
os.makedirs(data_folder, exist_ok=True)

smiles_file  = os.path.join(data_folder, "smiles.csv")
labels_file  = os.path.join(data_folder, "labels.csv")
matrix_path  = os.path.join(data_folder, "matrix.csv")
annot_path   = os.path.join(data_folder, "annotations.csv")
alerts_file  = os.path.join(data_folder, "alert_collection.csv")

target_cells = [
    'A2058','A2780','A375','A427','CAOV3','COLO320DM','DLD1','EFM192B','ES2',
    'HCT116','HT144','HT29','KPL1','LNCAP','LOVO','MDAMB436','MSTO','NCIH1650',
    'NCIH2122','NCIH23','NCIH460','NCIH520','OCUBM','OV90','OVCAR3','PA1','RKO',
    'RPMI7951','SKMEL30','SKMES1','SKOV3','SW620','SW837','T47D','UACC62',
    'UWB1289','UWB1289BRCA1','VCAP','ZR751'
]

In [2]:
ecfp_df   = generate_ecfp6_df(smiles_file)
phys_df   = generate_physico_df(smiles_file, labels_file)
toxico_df = generate_toxico_df(smiles_file, alerts_file)
expr_df   = process_cellline_matrix(matrix_path, annot_path, target_cells)

labels = pd.read_csv(labels_file)
labels.columns = labels.columns.str.strip()
if toxico_df.columns[-1] == 'Drug':
    toxico_df = toxico_df[['Drug'] + [c for c in toxico_df.columns if c != 'Drug']]

[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerator
[15:49:57] DEPRECATION WARNING: please use MorganGenerat

In [3]:
def make_AB(df, drug_col='Drug', prefix=''):
    A = df.copy().add_prefix(f"A_{prefix}")
    A = A.rename(columns={f"A_{prefix}{drug_col}": 'drug_a_name'})
    B = df.copy().add_prefix(f"B_{prefix}")
    B = B.rename(columns={f"B_{prefix}{drug_col}": 'drug_b_name'})
    return A, B

ecfp_A, ecfp_B = make_AB(ecfp_df, drug_col='Drug', prefix='ECFP_')
tox_A, tox_B   = make_AB(toxico_df, drug_col='Drug', prefix='TOX_')

In [4]:
df = labels.copy()
df = df.merge(ecfp_A, on='drug_a_name', how='left')
df = df.merge(ecfp_B, on='drug_b_name', how='left')
phys_df = phys_df.reset_index(drop=True)
df = pd.concat([df, phys_df], axis=1)
df = df.merge(tox_A, on='drug_a_name', how='left')
df = df.merge(tox_B, on='drug_b_name', how='left')
print('Drug features merged')

Drug features merged


In [5]:
genes_t = expr_df.T.reset_index().rename(columns={'index': 'cell_line'})
gene_cols = [c for c in genes_t.columns if c != 'cell_line']
genes_t[gene_cols] = genes_t[gene_cols].astype('float32')
top_n = 4000
gene_variances = genes_t[gene_cols].var(axis=0)
top_genes = gene_variances.sort_values(ascending=False).head(top_n).index
genes_t_top = genes_t[['cell_line'] + top_genes.tolist()]
chunk_size = 500
df_chunks = []
for start in range(0, genes_t_top.shape[0], chunk_size):
    chunk = genes_t_top.iloc[start:start+chunk_size]
    merged_chunk = df.merge(chunk, on='cell_line', how='left')
    df_chunks.append(merged_chunk)
df = pd.concat(df_chunks, ignore_index=True)
print('Gene features merged.')
print('Current shape:', df.shape)

Gene features merged.
Current shape: (23052, 10213)


In [6]:
out_pickle = os.path.join(data_folder, 'Feature_vectors_dataset.p.gz')
with gzip.open(out_pickle, 'wb') as f:
    pickle.dump(df, f)
print('Saved:', out_pickle)
df_zv = df.loc[:, df.nunique() > 1]
out_zv_pickle = os.path.join(data_folder, 'Feature_vectors_dataset_zeroVarRemoved.p.gz')
with gzip.open(out_zv_pickle, 'wb') as f:
    pickle.dump(df_zv, f)
print('Saved:', out_zv_pickle)

file_before_zv = out_pickle
file_after_zv  = out_zv_pickle
def get_pickle_shape(file_path):
    with gzip.open(file_path, 'rb') as f:
        df = pickle.load(f)
    return df.shape

shape_before = get_pickle_shape(file_before_zv)
shape_after  = get_pickle_shape(file_after_zv)
print('Shape before zero-variance filtering:', shape_before)
print('Shape after zero-variance filtering:', shape_after)

Saved: /mnt/c/Users/mayak/DeepSyn/feature_vector/dataset/Feature_vectors_dataset.p.gz
Saved: /mnt/c/Users/mayak/DeepSyn/feature_vector/dataset/Feature_vectors_dataset_zeroVarRemoved.p.gz
Shape before zero-variance filtering: (23052, 10213)
Shape after zero-variance filtering: (23052, 7094)


In [2]:
import os
import numpy as np
import pandas as pd
import pickle
import gzip
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import backend as K

In [3]:
norm = 'tanh'     
test_fold = 0
val_fold = 1
X_file = '/mnt/c/Users/mayak/DeepSyn/feature_vector/dataset/Feature_vectors_dataset_zeroVarRemoved.p.gz'
labels_file = '/mnt/c/Users/mayak/DeepSyn/feature_vector/dataset/labels.csv'

In [4]:
def normalize(X, means1=None, std1=None, means2=None, std2=None, feat_filt=None, norm='tanh_norm'):
    if std1 is None:
        std1 = np.nanstd(X, axis=0)
    if feat_filt is None:
        feat_filt = std1 != 0
    X = X[:, feat_filt]
    X = np.ascontiguousarray(X)
    if means1 is None:
        means1 = np.mean(X, axis=0)
    X = (X - means1) / std1[feat_filt]
    if norm == 'norm':
        return X, means1, std1, feat_filt
    elif norm == 'tanh':
        return np.tanh(X), means1, std1, feat_filt
    elif norm == 'tanh_norm':
        X = np.tanh(X)
        if means2 is None:
            means2 = np.mean(X, axis=0)
        if std2 is None:
            std2 = np.std(X, axis=0)
        X = (X - means2) / std2
        X[:, std2 == 0] = 0
        return X, means1, std1, means2, std2, feat_filt

In [5]:
with gzip.open(X_file, 'rb') as f:
    X = pickle.load(f)
labels = pd.read_csv(labels_file, index_col=0)

In [6]:
idx_tr = np.where((labels['fold'] != test_fold) & (labels['fold'] != val_fold))[0]
idx_val = np.where(labels['fold'] == val_fold)[0]
idx_train = np.where(labels['fold'] != test_fold)[0]
idx_test = np.where(labels['fold'] == test_fold)[0]

In [7]:
X_tr = X.iloc[idx_tr]
X_val = X.iloc[idx_val]
X_train = X.iloc[idx_train]
X_test = X.iloc[idx_test]
y_tr = labels.iloc[idx_tr]['synergy'].values
y_val = labels.iloc[idx_val]['synergy'].values
y_train = labels.iloc[idx_train]['synergy'].values
y_test = labels.iloc[idx_test]['synergy'].values
X_tr_np = X_tr.select_dtypes(include=[np.number]).to_numpy(dtype=np.float32)
X_val_np = X_val.select_dtypes(include=[np.number]).to_numpy(dtype=np.float32)
X_train_np = X_train.select_dtypes(include=[np.number]).to_numpy(dtype=np.float32)
X_test_np = X_test.select_dtypes(include=[np.number]).to_numpy(dtype=np.float32)

y_tr_np = y_tr.astype(np.float32)
y_val_np = y_val.astype(np.float32)
y_train_np = y_train.astype(np.float32)
y_test_np = y_test.astype(np.float32)



In [8]:
if norm == "tanh_norm":
    X_tr_np, mean, std, mean2, std2, feat_filt = normalize(X_tr_np, norm=norm)
    X_val_np, _, _, _, _, _ = normalize(X_val_np, mean, std, mean2, std2, feat_filt=feat_filt, norm=norm)
else:
    X_tr_np, mean, std, feat_filt = normalize(X_tr_np, norm=norm)
    X_val_np, _, _, _ = normalize(X_val_np, mean, std, feat_filt=feat_filt, norm=norm)
y_tr_np = y_tr_np.reshape(-1, 1)
y_val_np = y_val_np.reshape(-1, 1)

In [ ]:
import itertools
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
norm_options = ['norm', 'tanh', 'tanh_norm']
hidden_options = [
    [8192, 8192], [4096, 4096], [2048, 2048],
    [8192, 4096], [4096, 2048], [4096, 4096, 4096],
    [2048, 2048, 2048], [4096, 2048, 1024], [8192, 4096, 2048]]
lr_options = [1e-2, 1e-3, 1e-4, 1e-5]
dropout_options = [(0,0), (0.2,0.5)]
hyperparameter_grid = list(itertools.product(norm_options, hidden_options, lr_options, dropout_options))

best_val_loss = np.inf
best_params = None
for norm_type, hidden_layers, lr, (input_do, hidden_do) in hyperparameter_grid:
    if norm_type == "tanh_norm":
        X_tr_norm, mean, std, mean2, std2, feat_filt = normalize(X_tr_np.copy(), norm=norm_type)
        X_val_norm, _, _, _, _, _ = normalize(X_val_np.copy(), mean, std, mean2, std2, feat_filt=feat_filt, norm=norm_type)
    elif norm_type == "tanh":
        X_tr_norm, mean, std, feat_filt = normalize(X_tr_np.copy(), norm=norm_type)
        X_val_norm, _, _, _ = normalize(X_val_np.copy(), mean, std, feat_filt=feat_filt, norm=norm_type)
    else:
        X_tr_norm, mean, std, feat_filt = normalize(X_tr_np.copy(), norm=norm_type)
        X_val_norm, _, _, _ = normalize(X_val_np.copy(), mean, std, feat_filt=feat_filt, norm=norm_type)
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(units, input_shape=(X_tr_norm.shape[1],), activation='relu', kernel_initializer='he_normal'))
            if input_do > 0:
                model.add(Dropout(input_do))
        elif i == len(hidden_layers) - 1:
            model.add(Dense(units, activation='linear', kernel_initializer='he_normal'))
        else:
            model.add(Dense(units, activation='relu', kernel_initializer='he_normal'))
            if hidden_do > 0:
                model.add(Dropout(hidden_do))

    model.compile(loss='mean_squared_error', optimizer=SGD(learning_rate=lr, momentum=0.5))
    hist = model.fit(X_tr_norm, y_tr_np, validation_data=(X_val_norm, y_val_np), epochs=50, batch_size=64, verbose=0)
    val_loss = min(hist.history['val_loss'])
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = (norm_type, hidden_layers, lr, input_do, hidden_do)
print("Best hyperparameters:", best_params)
print("Validation loss:", best_val_loss)

/home/mayak/dsenv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/mayak/dsenv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-19 07:40:07.396998: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
